In [24]:
import os
import glob
import torch
import torch.nn as nn
import onnx

from finn.util.basic import make_build_dir
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper

# Instead of build_dataflow_cfg, we import the dataflow transforms:
from finn.transformation.fpgadataflow.convert_to_hls import ConvertToHLSLayer
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.export import ExportRTL
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer

# Basic FINN transforms
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.streamline import Streamline
import finn.transformation.streamline.absorb as absorb
from qonnx.transformation.general import (
    RemoveUnusedTensors,
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
)
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount

# Import your UNet and any helper
from split_unet import UNetSplit

class EncoderOnly(nn.Module):
    def __init__(self, unet_split):
        super().__init__()
        self.encoder = unet_split.encoder
    def forward(self, x):
        encoded_x, skip_list = self.encoder(x)
        return encoded_x

def remove_final_mul_op(model: ModelWrapper):
    graph = model.graph
    fout_name = graph.output[0].name
    last_node = None
    for node in graph.node:
        if fout_name in node.output:
            last_node = node
            break
    if last_node is not None and last_node.op_type == "Mul":
        mul_input = last_node.input[0]
        print("Removing final Mul node, rewiring output to:", mul_input)
        graph.output[0].name = mul_input
        keep_nodes = [n for n in graph.node if n != last_node]
        graph.ClearField("node")
        graph.node.extend(keep_nodes)
    else:
        print("No final Mul node found.")
    return model

def rename_final_output(model: ModelWrapper, new_name="final_out"):
    old_name = model.graph.output[0].name
    print(f"Renaming final output from {old_name} to {new_name}")
    model.graph.output[0].name = new_name
    # remove duplicates from input/value_info if they used old_name
    keep_inp = []
    for inp in model.graph.input:
        if inp.name != old_name:
            keep_inp.append(inp)
    del model.graph.input[:]
    model.graph.input.extend(keep_inp)

    new_vi = []
    for vi in model.graph.value_info:
        if vi.name != old_name:
            new_vi.append(vi)
    del model.graph.value_info[:]
    model.graph.value_info.extend(new_vi)
    return model

def main():
    build_dir = make_build_dir("manual_dataflow_build")
    MODEL_WEIGHTS = "./best_unet_weights.pth"

    # 1) Create the full U-Net, load weights
    unet_full = UNetSplit(1, 1)
    unet_full.load_state_dict(torch.load(MODEL_WEIGHTS, map_location="cpu"))
    unet_full.eval()

    # 2) Create encoder-only
    encoder_only = EncoderOnly(unet_full).eval()

    # 3) Export to ONNX, basic FINN transforms
    export_onnx_path = os.path.join(build_dir, "unet_encoder_export.onnx")
    dummy_in = torch.randn(1,1,128,128)
    export_qonnx(encoder_only, dummy_in, export_onnx_path)
    qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)

    model = ModelWrapper(export_onnx_path)
    model = model.transform(ConvertQONNXtoFINN())
    model = model.transform(InferShapes())
    model = model.transform(FoldConstants())
    model = model.transform(Streamline())
    model = model.transform(LowerConvsToMatMul())
    model = model.transform(ConvertBipolarMatMulToXnorPopcount())
    model = model.transform(Streamline())
    model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
    model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
    model = model.transform(InferDataLayouts())
    model = model.transform(RemoveUnusedTensors())
    model = model.transform(RemoveStaticGraphInputs())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())

    # 4) Remove final Mul, rename output
    model = remove_final_mul_op(model)
    model = rename_final_output(model, "final_out")
    streamlined_path = os.path.join(build_dir, "unet_encoder_streamlined.onnx")
    model.save(streamlined_path)
    print("Final streamlined, single-output model at:", streamlined_path)

    # 5) Manual dataflow pipeline (no partial partition)
    # Convert all layers to HLS
    model = model.transform(ConvertToHLSLayer())
    # set exec mode to "rtl" or "hls"
    model = model.transform(SetExecMode("rtl"))
    model.save(os.path.join(build_dir, "df_before_sim.onnx"))

    # Prepare C++Sim
    model = model.transform(PrepareCppSim())
    model.save(os.path.join(build_dir, "df_prepared_cppsim.onnx"))

    # HLS synth
    model = model.transform(HLSSynthIP())
    model.save(os.path.join(build_dir, "df_hls_synth.onnx"))

    # Create stitched IP
    model = model.transform(CreateStitchedIP(
        platform="Pynq-Z2",
        period_ns=10.0,
        generate_bitfile=True
    ))
    model.save(os.path.join(build_dir, "df_stitched.onnx"))

    # Export RTL, Make PYNQ driver
    model = model.transform(ExportRTL())
    model = model.transform(MakePYNQDriver(platform="Pynq-Z2"))

    final_path = os.path.join(build_dir, "df_final.onnx")
    model.save(final_path)
    print("Manual dataflow build complete, final onnx at:", final_path)
    print("Check build folder for bitfile + driver")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'finn.transformation.fpgadataflow.convert_to_hls'